# Anchor Workshop

Interactive notebook for defining, generating, reviewing, and exporting content anchors for AutoConcourse.

**Workflow:**
1. Load your topic config and subcategory definitions
2. Generate candidate anchors via LLM (or write them manually)
3. Review and edit each anchor
4. Export final anchors to JSON for pipeline use

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from autoconcourse.config import load_config
from autoconcourse.anchors import generate_anchors, load_anchors, save_anchors
from autoconcourse.generation import create_llm_client

## 1. Load Configuration

Point to your topic config file. This loads the subcategory definitions and LLM settings.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "healthcare_ch.yaml"  # <-- Change for your topic

config = load_config(CONFIG_PATH)
print(f"Topic: {config.topic.name}")
print(f"Languages: {config.topic.languages}")
print(f"Anchor model: {config.anchors.model}")

In [ ]:
# Load subcategory definitions
subcategories_path = config.resolve_path(config.anchors.subcategories_file)
with open(subcategories_path) as f:
    subcategories = json.load(f)

print(f"\n{len(subcategories)} subcategories:")
for sc in subcategories:
    print(f"  [{sc['id']}] {sc['name']}: {sc['description'][:80]}...")

## 2. Generate Anchor Candidates

Use the LLM to generate anchor paragraphs for each subcategory and language. You can also skip this and write anchors manually in step 3.

In [ ]:
client = create_llm_client(config)
anchors_df = generate_anchors(config, client=client)
print(f"Generated {len(anchors_df)} anchor texts")
anchors_df.head()

## 3. Review and Edit Anchors

Review each generated anchor. Edit in place or replace entirely.

In [ ]:
# Display anchors grouped by subcategory for review
for cat_id in sorted(anchors_df["category"].unique()):
    cat_anchors = anchors_df[anchors_df["category"] == cat_id]
    name = cat_anchors["subcategory_name"].iloc[0]
    print(f"\n{'='*80}")
    print(f"Subcategory {cat_id}: {name}")
    print(f"{'='*80}")
    for _, row in cat_anchors.iterrows():
        print(f"\n  [{row['language']}] {row['text'][:200]}...")

In [ ]:
# To edit a specific anchor, modify the DataFrame directly:
# anchors_df.loc[(anchors_df["category"] == 0) & (anchors_df["language"] == "de"), "text"] = "Your edited text here"

# To add a manual anchor:
# new_anchor = pd.DataFrame([{"category": 0, "text": "...", "language": "en", "subcategory_name": "..."}])
# anchors_df = pd.concat([anchors_df, new_anchor], ignore_index=True)

## 4. Export Anchors

Save the reviewed anchors. The pipeline will load them from the cache directory.

In [ ]:
output_path = config.cache_dir / "anchors.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
save_anchors(anchors_df, output_path)
print(f"Saved {len(anchors_df)} anchors to {output_path}")

# Also save as JSON for reference
json_path = config.resolve_path(f"data/{config.topic.id}/anchors.json")
json_path.parent.mkdir(parents=True, exist_ok=True)
save_anchors(anchors_df, json_path)
print(f"Saved JSON copy to {json_path}")